# Lista 2: Inteligência Artificial (UFAL)
## Questão 15: Ontologia OWL 2, representação do conhecimento e raciocínio

**Dupla:** Bruno Gomes de Barros Filho (21111360); Antônio Rodrigo Lima de Andrade Tenório (23210707)
**Disciplina:** ECOM031: Inteligência Artificial
**Professor:** Evandro de Barros Costa
**Período:** 2026.2

### Enunciado (Q15)
> Apresente e discuta um exemplo de aplicação de sua escolha que faça uso de ontologia com OWL 2.0,
> enfatizando a representação do conhecimento e raciocínio. Para isso, utilize a ferramenta protégé
> (ou alguma outra de sua preferência) e alguma máquina de inferência embutida nela.

**Aplicação:** a mesma ontologia de atividades físicas da Questão 9 (`ontologia_atividades_fisicas.owl`).
Aqui o foco sai do recomendador e vai para a lógica: que axiomas existem, em que Lógica de Descrição eles
estão, o que o reasoner deduz, o que ele não deduz por causa do mundo aberto e onde a linguagem para.

O Protégé é uma interface gráfica e não dá para registrá-la num notebook. Todos os passos abaixo usam o
HermiT que vem dentro do owlready2 (versão 1.3.8; o menu *Reasoner → HermiT* do Protégé 5.6 usa a 1.4.3,
mesmo grupo de Oxford e mesmo algoritmo) e o Pellet 2.3.1, que também tem plugin para o Protégé. Cada seção indica o equivalente na interface; o passo
a passo completo está no `README.md`.

In [1]:
import time
import warnings
from statistics import mean, stdev

import matplotlib.pyplot as plt
import pandas as pd
import rdflib
from owlready2 import OwlReadyInconsistentOntologyError, OwlReadyJavaError, ThingClass

import ontologia_atividades as oa
from figuras_ontologia import AZUL, LARANJA, TEXTO, TEXTO_2, _salvar, figura_hierarquias, pais_diretos
from raciocinio_owl import (carregar_ontologia, classificacao_hermit, classificacao_pellet,
                            consistente_pellet, explicar_pellet, raciocinar_hermit)
from sintaxe_owl import axiomas, dl, manchester

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 130)
world, onto = carregar_ontologia()

## 1. Representação: axiomas em Lógica de Descrição e em Manchester Syntax

A ontologia separa três partes:

* **TBox** (terminologia): classes, hierarquia e definições, como `UsuarioIdoso ≡ Usuario ⊓ ∃idade.integer[≥ 60]`;
* **RBox** (papéis): características e relações entre propriedades, como `parteDe` transitiva e as cadeias;
* **ABox** (asserções): fatos sobre indivíduos, como `gastoMET(corrida, 9.8)`.

A tabela traz os axiomas de TBox e RBox gerados a partir do próprio arquivo, nas duas notações. A coluna
Manchester é o que o Protégé mostra na aba *Description* da classe ou da propriedade.

In [2]:
tab = pd.DataFrame(axiomas(onto))
print(tab["tipo"].value_counts().to_string())
selecao = ["AtividadeVigorosa", "AtividadeAdequadaParaIdoso", "UsuarioIdoso", "UsuarioSemCondicaoDeSaude",
           "NivelDeImpacto", "naoDevePraticar", "contraindicadaPara", "sobrecarrega", "parteDe", "temImpacto",
           "AtividadeFisica", "Disj("]
tab[tab["DL"].apply(lambda s: any(k in s for k in selecao))
    & tab["tipo"].isin(["equivalência", "subclasse", "cadeia", "característica", "disjunção", "inversa"])
   ].drop_duplicates("DL").head(30)

tipo
subclasse         36
domínio           23
imagem            23
equivalência      22
característica    12
inversa            5
cadeia             5
disjunção          3


,tipo,DL,Manchester
0,subclasse,AtividadeFisica ⊑ =1 temImpacto.NivelDeImpacto,Class: AtividadeFisica SubClassOf: temImpacto exactly 1 NivelDeImpacto
1,subclasse,AtividadeFisica ⊑ =1 gastoMET.decimal,Class: AtividadeFisica SubClassOf: gastoMET exactly 1 decimal
2,subclasse,AtividadeFisica ⊑ ≥1 promove.Beneficio,Class: AtividadeFisica SubClassOf: promove min 1 Beneficio
3,subclasse,AtividadeFisica ⊑ ≥1 realizadaEm.Local,Class: AtividadeFisica SubClassOf: realizadaEm min 1 Local
4,equivalência,"NivelDeImpacto ≡ {impacto_baixo, impacto_moderado, impacto_alto}","Class: NivelDeImpacto EquivalentTo: {impacto_baixo, impacto_moderado, impacto_alto}"
7,subclasse,AtividadeAerobica ⊑ AtividadeFisica,Class: AtividadeAerobica SubClassOf: AtividadeFisica
8,subclasse,AtividadeDeForca ⊑ AtividadeFisica,Class: AtividadeDeForca SubClassOf: AtividadeFisica
9,subclasse,AtividadeDeFlexibilidade ⊑ AtividadeFisica,Class: AtividadeDeFlexibilidade SubClassOf: AtividadeFisica
10,subclasse,Esporte ⊑ AtividadeFisica,Class: Esporte SubClassOf: AtividadeFisica
17,equivalência,AtividadeDeBaixoImpacto ≡ AtividadeFisica ⊓ ∃temImpacto.{impacto_baixo},Class: AtividadeDeBaixoImpacto EquivalentTo: AtividadeFisica and (temImpacto value impacto_baixo)


A ABox de um indivíduo, em DL:

In [3]:
def abox(ind):
    linhas = [f"{t.name}({ind.name})" for t in ind.is_a if isinstance(t, ThingClass)]
    for p in list(onto.object_properties()) + list(onto.data_properties()):
        valores = getattr(ind, p.python_name)
        for v in (valores if isinstance(valores, list) else [valores] if valores is not None else []):
            linhas.append(f"{p.name}({ind.name}, {getattr(v, 'name', v)})")
    return linhas

print("\n".join(abox(onto.corrida)))
print()
print("\n".join(abox(onto.maria)))

AtividadeAerobica(corrida)
temImpacto(corrida, impacto_alto)
temNivelTecnico(corrida, nivel_basico)
realizadaEm(corrida, orla_de_ponta_verde)
realizadaEm(corrida, parque_municipal_de_maceio)
sobrecarrega(corrida, joelho)
sobrecarrega(corrida, tornozelo)
promove(corrida, gasto_calorico)
promove(corrida, condicionamento_cardiorrespiratorio)
gastoMET(corrida, 9.8)
custoMensal(corrida, 0.0)
minimoDeParticipantes(corrida, 1)

Usuario(maria)
temCondicao(maria, artrose_no_joelho)
temObjetivo(maria, socializar)
temObjetivo(maria, prevenir_quedas)
idade(maria, 67)
imc(maria, 27.0)
minutosAtivosPorSemana(maria, 60)
mesesDeExperiencia(maria, 0)
orcamentoMensal(maria, 200.0)
prefereAmbiente(maria, indiferente)


### 1.1 Expressividade: SROIQ(D)

OWL 2 DL corresponde à Lógica de Descrição **SROIQ(D)**. Cada letra aparece em algum axioma desta ontologia:

| Construtor | Letra | Exemplo na ontologia |
| :--- | :--- | :--- |
| ⊓, ⊔, ¬, ∃, ∀ (base ALC) e papéis transitivos | S | `parteDe` transitiva; `AtividadeAoArLivre ≡ AtividadeFisica ⊓ ∃realizadaEm.LocalAoArLivre` |
| Inclusão de papéis e cadeias | R | `temCondicao ∘ contraindica ⊑ naoDevePraticar` |
| Nominais (indivíduos dentro de classes) | O | `NivelDeImpacto ≡ {impacto_baixo, impacto_moderado, impacto_alto}`; `∃temImpacto.{impacto_baixo}` |
| Papéis inversos | I | `contraindica ≡ contraindicadaPara⁻` |
| Cardinalidade qualificada | Q | `AtividadeFisica ⊑ =1 temImpacto.NivelDeImpacto`; `temCondicao max 0 CondicaoDeSaude` |
| Tipos de dados com facetas | (D) | `AtividadeVigorosa ≡ AtividadeFisica ⊓ ∃gastoMET.decimal[≥ 6.0]` |

Propriedades funcionais (`temImpacto`, todas as de dados) são o caso `≤1` de Q.

## 2. Raciocínio: as tarefas clássicas

### 2.1 Consistência

Pergunta: existe pelo menos um modelo (uma interpretação que satisfaz todos os axiomas)? No Protégé, é o
que acontece ao clicar em *Start reasoner*; se não houver modelo, aparece a janela *Inconsistent ontology
explanation*.

In [4]:
resultado = {}
for nome, caminho in [("principal", oa.ARQUIVO_OWL), ("cópia inconsistente", oa.ARQUIVO_INCONSISTENTE),
                      ("cópia insatisfazível", oa.ARQUIVO_INSATISFAZIVEL)]:
    w, _ = carregar_ontologia(caminho)
    try:
        raciocinar_hermit(w)
        hermit = "consistente"
    except OwlReadyInconsistentOntologyError:
        hermit = "INCONSISTENTE"
    resultado[nome] = {"HermiT": hermit, "Pellet": "consistente" if consistente_pellet(caminho) else "INCONSISTENTE"}
pd.DataFrame(resultado).T

,HermiT,Pellet
principal,consistente,consistente
cópia inconsistente,INCONSISTENTE,INCONSISTENTE
cópia insatisfazível,consistente,consistente


In [5]:
print(explicar_pellet("--inconsistent", caminho=oa.ARQUIVO_INCONSISTENTE))

Axiom: Thing subClassOf Nothing
1)   Functional temImpacto
     corrida temImpacto impacto_alto
     DifferentIndividuals(impacto_alto
                          impacto_baixo
                          impacto_moderado)
     corrida type temImpacto value impacto_baixo


A justificativa tem quatro axiomas: `temImpacto` funcional, o fato declarado `temImpacto(corrida, impacto_alto)`,
a restrição `corrida ∈ ∃temImpacto.{impacto_baixo}` (o erro de cadastro) e a declaração de que os três níveis
são indivíduos diferentes. Sem o `DifferentIndividuals` não haveria contradição: como OWL não adota nomes
únicos, o reasoner concluiria que `impacto_alto` e `impacto_baixo` são o mesmo indivíduo.

### 2.2 Satisfatibilidade de classes

Uma ontologia pode ser consistente e ainda ter classes que nunca terão instâncias. A cópia
`ontologia_atividades_fisicas_insatisfazivel.owl` acrescenta
`AtividadeSuaveDeAltoImpacto ≡ AtividadeDeBaixoImpacto ⊓ AtividadeDeAltoImpacto`. No Protégé ela aparece em
vermelho, como subclasse de `owl:Nothing`, na hierarquia inferida.

In [6]:
w_ins, onto_ins = carregar_ontologia(oa.ARQUIVO_INSATISFAZIVEL)
print("Definição:", dl(onto_ins.AtividadeSuaveDeAltoImpacto.equivalent_to[0]))
raciocinar_hermit(w_ins)
print("HermiT, classes equivalentes a owl:Nothing:", [k.name for k in w_ins.inconsistent_classes() if k.name != "Nothing"])
print("Pellet:", classificacao_pellet(oa.ARQUIVO_INSATISFAZIVEL).insatisfaziveis)
print(explicar_pellet("--unsat", "AtividadeSuaveDeAltoImpacto", oa.ARQUIVO_INSATISFAZIVEL))

Definição: AtividadeDeBaixoImpacto ⊓ AtividadeDeAltoImpacto


HermiT, classes equivalentes a owl:Nothing: ['AtividadeSuaveDeAltoImpacto']


Pellet: {'AtividadeSuaveDeAltoImpacto'}


Axiom: AtividadeSuaveDeAltoImpacto subClassOf Nothing
1)   Functional temImpacto
     DifferentIndividuals(impacto_alto
                          impacto_baixo
                          impacto_moderado)
     AtividadeSuaveDeAltoImpacto equivalentTo AtividadeDeAltoImpacto
                                              and AtividadeDeBaixoImpacto
     AtividadeDeBaixoImpacto equivalentTo AtividadeFisica
                                          and temImpacto value impacto_baixo
     AtividadeDeAltoImpacto equivalentTo AtividadeFisica
                                         and temImpacto value impacto_alto


In [7]:
# Enquanto ninguém é instância da classe, a ontologia continua consistente.
# Basta declarar um indivíduo nela para virar inconsistência.
w_ins, onto_ins = carregar_ontologia(oa.ARQUIVO_INSATISFAZIVEL)
onto_ins.AtividadeSuaveDeAltoImpacto("atividade_impossivel")
try:
    raciocinar_hermit(w_ins)
except OwlReadyInconsistentOntologyError:
    print("Com um indivíduo na classe insatisfazível: ontologia inconsistente")

Com um indivíduo na classe insatisfazível: ontologia inconsistente


### 2.3 Subsunção e classificação

Subsunção é decidir se `C ⊑ D` vale em todos os modelos. Classificar é calcular todas as subsunções entre
classes nomeadas. Na ontologia, as classes definidas foram declaradas apenas como subclasses de
`AtividadeFisica` ou `Usuario`; o HermiT descobre o resto. No Protégé, compare as abas *Asserted* e *Inferred*
da hierarquia de classes.

In [8]:
w_cls, o_cls = carregar_ontologia()
declarada = pais_diretos(o_cls)
raciocinar_hermit(w_cls)
inferida = pais_diretos(o_cls)
figura_hierarquias(declarada, inferida, "q15_hierarquia_antes_depois.png")

def fecho(pais, k):
    return set().union(*({p} | fecho(pais, p) for p in pais.get(k, ())))

novas = [(k, p) for k in inferida for p in fecho(inferida, k) - fecho(declarada, k)]
pd.DataFrame(sorted(novas), columns=["subclasse", "superclasse inferida"])

,subclasse,superclasse inferida
0,AtividadeAdequadaParaIdoso,AtividadeAdequadaParaSedentario
1,AtividadeAdequadaParaIdoso,AtividadeDeBaixoImpacto
2,AtividadeAdequadaParaObesidade,AtividadeDeBaixoImpacto
3,AtividadeLeve,AtividadeAdequadaParaSedentario
4,AtividadeModerada,AtividadeAdequadaParaSedentario


![antes e depois](figures/q15_hierarquia_antes_depois.png)

Por que `AtividadeAdequadaParaIdoso ⊑ AtividadeAdequadaParaSedentario`? As duas exigem `gastoMET < 6`, e a primeira acrescenta baixo impacto; toda instância da primeira satisfaz a definição da segunda.

In [9]:
print(explicar_pellet("--subclass", "AtividadeAdequadaParaIdoso,AtividadeAdequadaParaSedentario"))
print()
print(explicar_pellet("--subclass", "AtividadeLeve,AtividadeAdequadaParaSedentario"))

Axiom: AtividadeAdequadaParaIdoso subClassOf AtividadeAdequadaParaSedentario
1)   AtividadeAdequadaParaIdoso equivalentTo AtividadeFisica
                                             and gastoMET some decimal[< 6.0]
                                             and temImpacto value impacto_baixo
     AtividadeAdequadaParaSedentario equivalentTo AtividadeFisica
                                                  and gastoMET some decimal[< 6.0]



Axiom: AtividadeLeve subClassOf AtividadeAdequadaParaSedentario
1)   AtividadeLeve equivalentTo AtividadeFisica
                                and gastoMET some decimal[< 3.0]
     AtividadeAdequadaParaSedentario equivalentTo AtividadeFisica
                                                  and gastoMET some decimal[< 6.0]


### 2.4 Realização (classificação de indivíduos)

Realizar é achar, para cada indivíduo, as classes mais específicas a que ele pertence. No Protégé, os tipos
inferidos aparecem em amarelo na aba *Individuals by class*.

In [10]:
linhas = []
_, o_dec = carregar_ontologia()
for ind in sorted(o_cls.individuals(), key=lambda i: i.name):
    antes = {t.name for t in o_dec[ind.name].INDIRECT_is_a if isinstance(t, ThingClass) and t.name != "Thing"}
    depois = {t.name for t in ind.INDIRECT_is_a if isinstance(t, ThingClass) and t.name != "Thing"}
    if depois - antes:
        linhas.append({"indivíduo": ind.name, "tipos antes": len(antes), "tipos depois": len(depois),
                       "novos tipos": ", ".join(sorted(depois - antes))})
realizacao = pd.DataFrame(linhas)
print(f"{len(realizacao)} indivíduos ganharam tipos; total de tipos novos: {(realizacao['tipos depois'] - realizacao['tipos antes']).sum()}")
realizacao

21 indivíduos ganharam tipos; total de tipos novos: 111


,indivíduo,tipos antes,tipos depois,novos tipos
0,alongamento,2,9,"AtividadeAdequadaParaIdoso, AtividadeAdequadaParaIniciante, AtividadeAdequadaParaSedentario, AtividadeAoArLivre, AtividadeDeBa..."
1,beach_tennis,2,5,"AtividadeAoArLivre, AtividadeEmGrupo, AtividadeVigorosa"
2,caminhada,2,10,"AtividadeAdequadaParaIdoso, AtividadeAdequadaParaIniciante, AtividadeAdequadaParaObesidade, AtividadeAdequadaParaSedentario, A..."
3,ciclismo_na_orla,2,7,"AtividadeAdequadaParaObesidade, AtividadeAoArLivre, AtividadeDeBaixoImpacto, AtividadeGratuita, AtividadeVigorosa"
4,corrida,2,7,"AtividadeAdequadaParaIniciante, AtividadeAoArLivre, AtividadeDeAltoImpacto, AtividadeGratuita, AtividadeVigorosa"
5,crossfit,2,5,"AtividadeDeAltoImpacto, AtividadeEmAmbienteFechado, AtividadeVigorosa"
6,futebol_society,2,6,"AtividadeAoArLivre, AtividadeDeAltoImpacto, AtividadeEmGrupo, AtividadeVigorosa"
7,hidroginastica,2,10,"AtividadeAdequadaParaIdoso, AtividadeAdequadaParaIniciante, AtividadeAdequadaParaObesidade, AtividadeAdequadaParaSedentario, A..."
8,joao,1,4,"UsuarioComObesidade, UsuarioIniciante, UsuarioSedentario"
9,lucas,1,2,UsuarioSemCondicaoDeSaude


### 2.5 Relações implícitas entre indivíduos

Além de tipos, o HermiT devolve valores de propriedades que decorrem das cadeias, das inversas e da
transitividade. No Protégé isso aparece em amarelo na aba *Object property assertions* do indivíduo (é
preciso marcar *Object property assertions* em *Reasoner → Configure*).

In [11]:
pares = [(u.name, a.name) for u in o_cls.Usuario.instances() for a in u.naoDevePraticar]
print(len(pares), "pares naoDevePraticar inferidos:", pares[:6], "...")
print("Partes sobrecarregadas pelo CrossFit (declaradas: coluna_lombar, joelho, ombro):",
      sorted(p.name for p in o_cls.crossfit.sobrecarrega))

15 pares naoDevePraticar inferidos: [('maria', 'volei_de_praia'), ('maria', 'futebol_society'), ('maria', 'crossfit'), ('maria', 'corrida'), ('maria', 'zumba'), ('joao', 'futebol_society')] ...
Partes sobrecarregadas pelo CrossFit (declaradas: coluna_lombar, joelho, ombro): ['coluna_lombar', 'coluna_vertebral', 'joelho', 'membro_inferior', 'membro_superior', 'ombro', 'sistema_cardiovascular', 'tronco']


## 3. Hipótese de mundo aberto (OWA)

Em OWL, a ausência de um fato não o torna falso. A classe `UsuarioSemCondicaoDeSaude ≡ Usuario ⊓ ≤0 temCondicao.CondicaoDeSaude`
deixa isso visível. Carla e Lucas não têm nenhuma condição cadastrada; a diferença é que, para Lucas, a
ontologia afirma que ele tem zero condições (axioma de fechamento).

In [12]:
print("Classificados pelo HermiT como UsuarioSemCondicaoDeSaude:",
      [u.name for u in o_cls.UsuarioSemCondicaoDeSaude.instances()])

g = rdflib.Graph().parse(oa.ARQUIVO_OWL, format="xml")
consulta = """
PREFIX : <http://ic.ufal.br/ecom031/lista2/atividades-fisicas#>
SELECT ?u WHERE { ?u a :Usuario . FILTER NOT EXISTS { ?u :temCondicao ?c } }"""
print("SPARQL com FILTER NOT EXISTS (mundo fechado sobre os dados):",
      sorted(str(r.u).split("#")[1] for r in g.query(consulta)))

Classificados pelo HermiT como UsuarioSemCondicaoDeSaude: ['lucas']


SPARQL com FILTER NOT EXISTS (mundo fechado sobre os dados): ['carla', 'lucas']


A consulta SPARQL responde como um banco de dados: quem não tem a tripla, não tem condição. O reasoner só
classifica Lucas, porque Carla pode ter condições que não foram informadas. Para fechar o mundo de Carla
basta o mesmo axioma (no Protégé: em *Types* do indivíduo, adicionar `temCondicao max 0 CondicaoDeSaude`):

In [13]:
w_owa, o_owa = carregar_ontologia()
o_owa.carla.is_a.append(o_owa.temCondicao.max(0, o_owa.CondicaoDeSaude))
print("Axioma acrescentado:", manchester(o_owa.carla.is_a[-1]))
raciocinar_hermit(w_owa)
print("Agora:", [u.name for u in o_owa.UsuarioSemCondicaoDeSaude.instances()])

Axioma acrescentado: temCondicao max 0 CondicaoDeSaude


Agora: ['carla', 'lucas']


O mesmo vale para restrições de cardinalidade, que em OWL não funcionam como restrições de integridade.
`AtividadeFisica ⊑ ≥1 realizadaEm.Local` não obriga ninguém a cadastrar o local: o reasoner supõe que existe
algum local desconhecido. Uma atividade nova sem local é aceita e classificada pelo que se sabe dela.

In [14]:
w_rem, o_rem = carregar_ontologia()
remo = o_rem.AtividadeFisica("remo_indoor")
remo.temImpacto, remo.gastoMET = o_rem.impacto_baixo, 7.0
raciocinar_hermit(w_rem)
print("Sem local cadastrado -> consistente. Tipos inferidos:", sorted(k.name for k in remo.is_a))
print("É AtividadeAoArLivre?", remo in o_rem.AtividadeAoArLivre.instances(),
      "| É AtividadeEmAmbienteFechado?", remo in o_rem.AtividadeEmAmbienteFechado.instances())

w_rem, o_rem = carregar_ontologia()
remo = o_rem.AtividadeFisica("remo_indoor")
remo.temImpacto, remo.gastoMET = o_rem.impacto_baixo, 7.0
remo.is_a.append(o_rem.realizadaEm.max(0, o_rem.Local))
try:
    raciocinar_hermit(w_rem)
except OwlReadyInconsistentOntologyError:
    print("Fechando com 'realizadaEm max 0 Local': inconsistente com 'realizadaEm min 1 Local'")

Sem local cadastrado -> consistente. Tipos inferidos: ['AtividadeAdequadaParaObesidade', 'AtividadeVigorosa']
É AtividadeAoArLivre? False | É AtividadeEmAmbienteFechado? False


Fechando com 'realizadaEm max 0 Local': inconsistente com 'realizadaEm min 1 Local'


O recomendador da Q9 usa essa distinção: quando o usuário responde "nenhuma" condição, o sistema
acrescenta o fechamento; quando ele não responde, nada é assumido e o relatório avisa.

## 4. HermiT versus Pellet

Os dois são reasoners completos para OWL 2 DL, com algoritmos diferentes: o HermiT usa hipertableau e o
Pellet usa tableau com otimizações para ABox e explicações. A comparação abaixo roda cada um 5 vezes sobre os
mesmos arquivos. Os tempos incluem iniciar a JVM, que domina para uma ontologia deste tamanho; servem para
comparar ordem de grandeza, não o algoritmo.

Observação de ambiente: o `sync_reasoner_pellet` do owlready2 0.51 falha no Java 21 porque três classes do jar
`jena-arq-fixed` foram compiladas para Java 25. O Pellet foi chamado pela linha de comando com
`--loader OWLAPIv3`, que não usa essas classes (ver `raciocinio_owl.py`). Por esse caminho o Pellet faz
consistência, classificação, realização e explicações, mas não exporta todos os valores de propriedades
inferidos; esses foram conferidos fato a fato com `explain --property-value`.

In [15]:
N = 5
tempos = {}
for motor, funcao in [("HermiT", classificacao_hermit), ("Pellet", classificacao_pellet)]:
    execucoes = [funcao() for _ in range(N)]
    tempos[(motor, "classificação + realização")] = [r.segundos for r in execucoes]
    ultimo = execucoes[-1]
    if motor == "HermiT":
        hermit = ultimo
    else:
        pellet = ultimo

def tempo_consistencia_hermit():
    w, _ = carregar_ontologia(oa.ARQUIVO_INCONSISTENTE)
    t0 = time.perf_counter()
    try:
        raciocinar_hermit(w)
    except OwlReadyInconsistentOntologyError:
        pass
    return time.perf_counter() - t0

def tempo_consistencia_pellet():
    t0 = time.perf_counter()
    consistente_pellet(oa.ARQUIVO_INCONSISTENTE)
    return time.perf_counter() - t0

tempos[("HermiT", "consistência (cópia inconsistente)")] = [tempo_consistencia_hermit() for _ in range(N)]
tempos[("Pellet", "consistência (cópia inconsistente)")] = [tempo_consistencia_pellet() for _ in range(N)]
resumo = pd.DataFrame([{"reasoner": m, "tarefa": t, "média (s)": round(mean(v), 3), "desvio (s)": round(stdev(v), 3)}
                       for (m, t), v in tempos.items()])
resumo

,reasoner,tarefa,média (s),desvio (s)
0,HermiT,classificação + realização,0.880,0.057
1,Pellet,classificação + realização,1.085,0.045
2,HermiT,consistência (cópia inconsistente),0.452,0.037
3,Pellet,consistência (cópia inconsistente),0.696,0.053


In [16]:
print("Mesma hierarquia inferida:", hermit.superclasses == pellet.superclasses)
print("Mesmos tipos para todos os indivíduos:", hermit.tipos == pellet.tipos)
print("Classes comparadas:", len(hermit.superclasses), "| indivíduos comparados:", len(hermit.tipos),
      "| pares (indivíduo, classe):", sum(map(len, hermit.tipos.values())))

def pellet_justifica(fato):
    return "\n1)" in explicar_pellet("--property-value", fato)

ok = all(pellet_justifica(f"maria,naoDevePraticar,{a.name}") for a in o_cls.maria.naoDevePraticar)
print("Pellet também justifica os", len(o_cls.maria.naoDevePraticar), "pares naoDevePraticar de Maria deduzidos pelo HermiT:", ok)
print("Controle negativo, maria naoDevePraticar caminhada (não deduzido):", pellet_justifica("maria,naoDevePraticar,caminhada"))

Mesma hierarquia inferida: True
Mesmos tipos para todos os indivíduos: True
Classes comparadas: 39 | indivíduos comparados: 66 | pares (indivíduo, classe): 211


Pellet também justifica os 5 pares naoDevePraticar de Maria deduzidos pelo HermiT: True


Controle negativo, maria naoDevePraticar caminhada (não deduzido): False


In [17]:
tarefas = list(dict.fromkeys(t for _, t in tempos))
fig, ax = plt.subplots(figsize=(9, 3.6))
largura = 0.36
for desloc, (motor, cor) in zip((-largura / 2, largura / 2), [("HermiT", AZUL), ("Pellet", LARANJA)]):
    medias = [mean(tempos[(motor, t)]) for t in tarefas]
    desvios = [stdev(tempos[(motor, t)]) for t in tarefas]
    ys = [i + desloc for i in range(len(tarefas))]
    ax.barh(ys, medias, height=largura - 0.04, xerr=desvios, color=cor, label=motor,
            error_kw=dict(ecolor=TEXTO_2, lw=1))
    for y, m in zip(ys, medias):
        ax.text(m + max(desvios) + 0.03, y, f"{m:.2f} s", va="center", fontsize=9, color=TEXTO)
ax.set_yticks(range(len(tarefas)), tarefas)
ax.invert_yaxis()
ax.set_xlabel(f"tempo de parede em segundos (média de {N} execuções, inclui a JVM)", color=TEXTO_2)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e4e3df", lw=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
ax.set_title("HermiT x Pellet na ontologia de atividades físicas", loc="left")
_salvar(fig, "q15_tempos_hermit_pellet.png")

PosixPath('/home/antonio/Documentos/ia/lista2-ia-ufal/src/q09_q15_ontologia/figures/q15_tempos_hermit_pellet.png')

![tempos](figures/q15_tempos_hermit_pellet.png)

## 5. Limites da representação

**Decidibilidade.** Raciocinar em SROIQ(D) é decidível (N2ExpTime-completo no pior caso). Para manter isso,
OWL 2 DL impõe restrições globais: cadeias precisam ser regulares e propriedades não simples (transitivas ou
definidas por cadeia) não podem aparecer em restrições de cardinalidade. O HermiT recusa a ontologia se a
regra for violada:

In [18]:
w_lim, o_lim = carregar_ontologia()
with o_lim:
    class AtividadeLocalizada(o_lim.AtividadeFisica):
        equivalent_to = [o_lim.AtividadeFisica & o_lim.sobrecarrega.max(1, o_lim.ParteDoCorpo)]
try:
    raciocinar_hermit(w_lim)
except OwlReadyJavaError as erro:
    print(str(erro).splitlines()[1][:260])

Exception in thread "main" java.lang.IllegalArgumentException: Non-simple property '<http://ic.ufal.br/ecom031/lista2/atividades-fisicas#sobrecarrega>' or its inverse appears in the cardinality restriction 'ObjectMinCardinality(2 <http://ic.ufal.br/ecom031/lis


**Sem aritmética.** OWL compara um valor com constantes (`gastoMET ≥ 6`), mas não calcula. O IMC é calculado
em Python e gravado como dado; também não é possível escrever "custo da atividade ≤ orçamento do usuário",
porque isso compara dois valores de indivíduos diferentes. Essa comparação ficou no recomendador.

**Sem variáveis compartilhadas.** A regra "se o usuário é idoso e a atividade é adequada para idosos, então
recomendar" junta dois indivíduos por condições de classe. Em SWRL seria
`UsuarioIdoso(?u) ∧ AtividadeAdequadaParaIdoso(?a) → recomendavelPara(?a, ?u)`. Não usei SWRL: regras
gerais tornam o raciocínio indecidível e só a forma DL-safe é tratada pelos reasoners. A solução adotada
mantém a ligação perfil → tipo de atividade como anotação (`exigeAtividadeDoTipo`) e deixa a junção para o
código, que só lê inferências. As cadeias de propriedades resolvem os casos que são composição de relações
(`temCondicao ∘ contraindica`), sem precisar de regras.

**Mundo aberto como custo.** Cardinalidades e domínios não validam dados. Para checar, por exemplo, se toda
atividade tem local cadastrado, é preciso uma consulta SPARQL ou SHACL por fora do reasoner.